In [1]:
def load_data():

    import pandas as pd

    SENTENCES = "../data/sentences.csv.zip"

    dataframe = pd.read_csv(
        SENTENCES,
        index_col=False,
        compression="zip",
    )

    data = dataframe.phrase
    target = dataframe.target

    return data, target

In [2]:

def make_train_test_split(x, y):

    from sklearn.model_selection import train_test_split

    (x_train, x_test, y_train, y_test) = train_test_split(
        x,
        y,
        test_size=0.25,
        random_state=123456,
    )
    return x_train, x_test, y_train, y_test
  
  
  

In [3]:

def make_pipeline(estimator):

    from sklearn.feature_extraction.text import CountVectorizer, TfidfTransformer
    from sklearn.pipeline import Pipeline

    vectorizer = CountVectorizer(
        lowercase=True,
        analyzer="word",
        token_pattern=r"\b[a-zA-Z]\w+\b",
        stop_words="english",
    )

    transformer = TfidfTransformer()

    pipeline = Pipeline(
        steps=[
            ("vectorizer", vectorizer),
            ("transformer", transformer),
            ("estimator", estimator),
        ],
        verbose=False,
    )

    return pipeline

In [4]:
def make_grid_search(estimator, param_grid, cv=5):

    from sklearn.model_selection import GridSearchCV

    grid_search = GridSearchCV(
        estimator=estimator,
        param_grid=param_grid,
        cv=cv,
        scoring="balanced_accuracy",
    )

    return grid_search
  

In [5]:

def load_estimator():

    import os
    import pickle

    OUTPUT_FOLDER = "../submission"
    ESTIMATOR_PATH = f"{OUTPUT_FOLDER}/estimator.pkl"

    if not os.path.exists(ESTIMATOR_PATH):
        return None

    with open(ESTIMATOR_PATH, "rb") as file:
        estimator = pickle.load(file)

    return estimator

In [7]:

def save_estimator(estimator, x_test, y_test):

    import pickle

    from sklearn.metrics import balanced_accuracy_score

    OUTPUT_FOLDER = "../submission"
    ESTIMATOR_PATH = f"{OUTPUT_FOLDER}/estimator.pkl"

    best_estimator = load_estimator()

    if best_estimator is not None:

        saved_balanced_accuracy = balanced_accuracy_score(
            y_true=y_test, y_pred=best_estimator.predict(x_test)
        )

        current_balanced_accuracy = balanced_accuracy_score(
            y_true=y_test, y_pred=estimator.predict(x_test)
        )

        if current_balanced_accuracy < saved_balanced_accuracy:
            estimator = best_estimator

    with open(ESTIMATOR_PATH, "wb") as file:
        pickle.dump(estimator, file)
        

In [8]:
def train_logistic_regression():

    from sklearn.linear_model import LogisticRegression

    data, target = load_data()

    x_train, x_test, y_train, y_test = make_train_test_split(
        x=data,
        y=target,
    )

    pipeline = make_pipeline(
        estimator=LogisticRegression(max_iter=1000),
    )

    estimator = make_grid_search(
        estimator=pipeline,
        param_grid={
            "transformer__norm": ["l1", "l2", None],
            "transformer__use_idf": [True, False],
            "transformer__smooth_idf": [True, False],
        },
        cv=5,
    )

    estimator.fit(x_train, y_train)

    save_estimator(estimator, x_test, y_test)


train_logistic_regression()